# Chapter 2 Practical 02: TF-IDF Movie Recommender

Learning objectives:
- Clean and combine text metadata.
- Build Bag-of-Words and TF-IDF item vectors.
- Compute a cosine similarity matrix.
- Explain recommendations with shared terms.

Slide connection: Bag-of-Words, TF-IDF, vector normalization, cosine similarity, ranking, and Top-N recommendation.


Load the small Chapter 2 movie dataset. It includes titles, genres, descriptions, and keywords.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")
if not (DATA_DIR / "movies_chapter2.csv").exists():
    DATA_DIR = Path("../data")
if not (DATA_DIR / "movies_chapter2.csv").exists():
    DATA_DIR = Path("chapter_02_content_based/data")

movies = pd.read_csv(DATA_DIR / "movies_chapter2.csv")
movies.head()


We combine several text fields. This gives the recommender more content evidence than title or genre alone.


In [ ]:
import re
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s-]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

movies["combined_text"] = (
    movies["title"] + " " +
    movies["genres"].str.replace("|", " ", regex=False) + " " +
    movies["director"] + " " +
    movies["description"] + " " +
    movies["keywords"]
).apply(clean_text)

movies[["title", "combined_text"]].head()


Bag-of-Words counts words. Common words can dominate because each word is weighted mostly by frequency.


In [ ]:
count_vectorizer = CountVectorizer(stop_words="english")
bow_matrix = count_vectorizer.fit_transform(movies["combined_text"])

bow_preview = pd.DataFrame(
    bow_matrix.toarray(),
    columns=count_vectorizer.get_feature_names_out(),
    index=movies["title"],
)
bow_preview.iloc[:5, :12]


TF-IDF lowers the weight of terms that appear in many movies and raises distinctive terms.


In [ ]:
tfidf_vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf_vectorizer.fit_transform(movies["combined_text"])

tfidf_preview = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=movies["title"],
)
tfidf_preview.iloc[:5, :12].round(2)


The similarity matrix compares every movie with every other movie.


In [ ]:
bow_similarity = cosine_similarity(bow_matrix)
tfidf_similarity = cosine_similarity(tfidf_matrix)

pd.DataFrame(tfidf_similarity, index=movies["title"], columns=movies["title"]).round(2)


This function returns Top-N similar movies and shows which TF-IDF terms are shared with the input movie.


In [ ]:
def shared_terms(input_idx, other_idx, matrix, vectorizer, top_terms=6):
    feature_names = np.array(vectorizer.get_feature_names_out())
    input_weights = matrix[input_idx].toarray().ravel()
    other_weights = matrix[other_idx].toarray().ravel()
    shared = np.minimum(input_weights, other_weights)
    best = shared.argsort()[::-1][:top_terms]
    return ", ".join(feature_names[i] for i in best if shared[i] > 0)

def recommend_similar(title, similarity_matrix, matrix, vectorizer, n=5):
    idx = movies.index[movies["title"].eq(title)][0]
    scores = list(enumerate(similarity_matrix[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    rows = []
    for other_idx, score in scores[1:n+1]:
        rows.append({
            "input_movie": title,
            "recommended_movie": movies.loc[other_idx, "title"],
            "similarity_score": round(float(score), 3),
            "shared_terms_features": shared_terms(idx, other_idx, matrix, vectorizer),
        })
    return pd.DataFrame(rows)

recommend_similar("Interstellar", tfidf_similarity, tfidf_matrix, tfidf_vectorizer)


Compare Bag-of-Words and TF-IDF. The rankings may be similar, but TF-IDF usually gives cleaner emphasis to distinctive content.


In [ ]:
bow_results = recommend_similar("Interstellar", bow_similarity, bow_matrix, count_vectorizer, n=5)
tfidf_results = recommend_similar("Interstellar", tfidf_similarity, tfidf_matrix, tfidf_vectorizer, n=5)

comparison = bow_results[["recommended_movie", "similarity_score"]].rename(columns={"similarity_score": "bow_score"})
comparison["tfidf_movie"] = tfidf_results["recommended_movie"]
comparison["tfidf_score"] = tfidf_results["similarity_score"]
comparison


## What did we learn?

- Bag-of-Words creates count vectors from text.
- TF-IDF keeps the vector idea but gives more weight to distinctive terms.
- Cosine similarity turns text vectors into a ranked recommendation list.

Exercises:
1. Try the recommender with `Toy Story` or `Titanic`.
2. Add a new keyword to one movie and check whether the ranking changes.
